# 05 LightGBM

## Set-Up (Imports and Loading Data)

In [13]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgbm

import sys
sys.path.append('../src')

from load_clean_data import load_clean_data
from backtest import get_cutoffs, split_data_by_cutoff, run_backtest
from baselines import add_all_baselines
from evaluation import score_forecasts, merge_backtests
from features import build_features, FEATURE_COLUMNS
from models import add_lgbm_forecast, LGBM_PARAMS

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
df = load_clean_data()
df_with_features = build_features(df)

## Train Test Split

In [15]:
train, test = split_data_by_cutoff(df_with_features, '2015-02-02', 7)

## Run LGBM (Single Cutoff)

In [16]:
results = add_lgbm_forecast(train, test, '2015-02-02')

In [17]:
print(len(results))
print(results['lgbm_forecast'].describe())

7805
count     7805.000000
mean      6897.353473
std       4031.054958
min          0.000000
25%       4965.102391
50%       7006.734257
75%       9149.214288
max      27422.356873
Name: lgbm_forecast, dtype: float64


### Compare against baselines

In [18]:
results = add_all_baselines(train, results, '2015-02-02')

scores = score_forecasts(results, ['naive_baseline', 'avg_28d_baseline', 'avg_all_baseline', 'lgbm_forecast'])

In [19]:
scores.sort_values(by='WAPE').style.hide(axis='index').format({ 'WAPE': '{:.2f}%' }, precision=2)

Model,RMSE,MAE,WAPE
avg_28d_baseline,1128.94,711.88,10.99%
naive_baseline,1098.63,711.96,10.99%
lgbm_forecast,1167.04,764.01,11.80%
avg_all_baseline,1261.37,887.79,13.71%


## Run LGBM (All Cutoffs)

In [20]:
FORECAST = False

if FORECAST:
    cutoffs = get_cutoffs(df['Date'].max(), 7)
    bt = run_backtest(df_with_features, cutoffs, 7, add_lgbm_forecast)

    bt = bt[['Store', 'Date', 'Sales', 'cutoff', 'day_ahead', 'lgbm_forecast']]

    bt.to_csv('../results/backtests/lgbm_backtest.csv', index=False, )

else: 
    bt = pd.read_csv('../results/backtests/lgbm_backtest.csv', parse_dates=['Date', 'cutoff'], low_memory=False)

In [21]:
SAVE_SCORES = True

if SAVE_SCORES:
    scores = score_forecasts(bt, ['lgbm_forecast'])
    scores.to_csv('../results/lgbm_scores.csv', index=False)

In [ ]:
combined_bt = merge_backtests(['baselines', 'xgb', 'lgbm'])

forecasts = ['naive_baseline', 'avg_28d_baseline', 'avg_all_baseline', 'xgboost_forecast', 'lgbm_forecast']

combined_scores = score_forecasts(combined_bt, forecasts)

model_names = {
    'naive_baseline': 'Naive Baseline',
    'avg_28d_baseline': '28-Day Average Baseline',
    'avg_all_baseline': 'Full History Average Baseline',
    'xgboost_forecast': 'XGBoost Forecast', 
    'lgbm_forecast': 'LightGBM Forecast'
}

relabelled_scores = combined_scores.copy()
relabelled_scores['Model'] = relabelled_scores['Model'].replace(model_names)

relabelled_scores.sort_values(by='WAPE').style.hide(axis='index').format({ 'WAPE': '{:.2f}%' }, precision=2)

Model,RMSE,MAE,WAPE
XGBoost Forecast,1001.30,604.95,10.33%
LightGBM Forecast,1035.76,619.47,10.57%
Full History Average Baseline,2029.48,1261.49,21.53%
28-Day Average Baseline,2127.55,1326.10,22.63%
Naive Baseline,3206.68,2112.59,36.06%
